## Example 1: Basic HTML Parsing and Tree Building

In [ ]:
import html5lib

# Example 1: Basic HTML parsing
print("Example 1: Basic HTML parsing")

# Sample HTML content
html_content = """
<html>
<head><title>Sample Page</title></head>
<body>
    <h1>Welcome</h1>
    <p>This is a <strong>sample</strong> paragraph.</p>
    <ul><li>Item 1</li><li>Item 2</li></ul>
</body>
</html>
"""

# Parse HTML using html5lib
document = html5lib.parse(html_content)
print(f"Document parsed: {type(document).__name__}")
print(f"Root tag: {document.tag}")

# Parse HTML fragment
fragment = html5lib.parseFragment("<p>Hello <em>world</em>!</p>")
print(f"Fragment parsed: {len(list(fragment))} elements")

# The default tree is xml.etree; elements live in the XHTML namespace
NS = "{http://www.w3.org/1999/xhtml}"
assert document.tag == f"{NS}html"
assert document.find(f".//{NS}title").text == "Sample Page"
assert [li.text for li in document.iter(f"{NS}li")] == ["Item 1", "Item 2"]
assert [child.tag for child in fragment] == [f"{NS}p"]

## Example 2: HTML serialization and output

In [ ]:
import html5lib

# Example 2: HTML serialization and output
print("Example 2: HTML serialization and output")

# Parse HTML document
html_content = """
<html>
<head><title>Test</title></head>
<body>
    <h1>Hello World</h1>
    <p>This is a <strong>test</strong> paragraph.</p>
</body>
</html>
"""

# Parse the HTML
document = html5lib.parse(html_content)
print(f"Document parsed: {type(document).__name__}")

# Serialize back to HTML
serialized_html = html5lib.serialize(document)
print(f"Serialized HTML type: {type(serialized_html)}")
print(f"Serialized length: {len(serialized_html)} characters")

# Show first part of serialized HTML
preview = serialized_html[:100] if len(serialized_html) > 100 else serialized_html
print(f"HTML preview: {preview}...")

assert isinstance(serialized_html, str)
assert "<title>Test</title>" in serialized_html
assert "<p>This is a <strong>test</strong> paragraph." in serialized_html
# optional end tags are omitted by default
assert html5lib.serialize(html5lib.parseFragment("<p>Hello <em>world</em>!</p>")) == "<p>Hello <em>world</em>!"

## Example 3: HTML parser with different tree builders

In [ ]:
import html5lib

# Example 3: HTML parser with different tree builders
print("Example 3: HTML parser with different tree builders")

html_content = "<div>Hello <span>world</span>!</div>"

# Get available tree builders
tree_builder = html5lib.getTreeBuilder("etree")
print(f"Tree builder: {type(tree_builder).__name__}")

# Create parser with specific tree builder
parser = html5lib.HTMLParser(tree=tree_builder)
print(f"Parser created: {type(parser).__name__}")

# Parse HTML with custom parser
document = parser.parse(html_content)
print(f"Parsed document: {type(document).__name__}")

# Parse fragment with same parser
fragment = parser.parseFragment("<p>Fragment <em>text</em></p>")
print(f"Fragment parsed: {type(fragment)}")
print(f"Fragment elements: {len(list(fragment))}")

NS = "{http://www.w3.org/1999/xhtml}"
assert [child.tag for child in fragment] == [f"{NS}p"]
assert document.find(f".//{NS}span").text == "world"

# The same markup through the xml.dom.minidom tree builder
dom_parser = html5lib.HTMLParser(tree=html5lib.getTreeBuilder("dom"))
dom_document = dom_parser.parse(html_content)
assert dom_document.getElementsByTagName("span")[0].firstChild.data == "world"

## Example 4: Malformed HTML parsing and error handling

In [ ]:
import html5lib

# Example 4: Malformed HTML parsing and error handling
print("Example 4: Malformed HTML parsing and error handling")

# Malformed HTML with various issues
malformed_html = """
<html>
<head>
    <title>Broken HTML</title>
    <meta charset="utf-8">
</head>
<body>
    <h1>Unclosed header
    <p>Paragraph with <strong>unclosed tag
    <ul>
        <li>Item 1
        <li>Item 2</li>
    </ul>
    <div class="unclosed">
        Content without closing div
</body>
"""

# Parse malformed HTML - html5lib will fix errors automatically
document = html5lib.parse(malformed_html)
print(f"Malformed HTML parsed successfully: {type(document).__name__}")

# Serialize to see how html5lib corrected the HTML
corrected_html = html5lib.serialize(document)
print(f"Corrected HTML length: {len(corrected_html)} characters")

# Check if common elements are present after correction
corrected_str = ''.join(corrected_html) if isinstance(corrected_html, list) else corrected_html
has_title = 'title' in corrected_str.lower()
has_body = 'body' in corrected_str.lower()
print(f"Contains title tag: {has_title}")
print(f"Contains body tag: {has_body}")

# The parser repairs the tree the way browsers do
NS = "{http://www.w3.org/1999/xhtml}"
assert [li.text.strip() for li in document.iter(f"{NS}li")] == ["Item 1", "Item 2"]
assert document.find(f".//{NS}div").get("class") == "unclosed"
assert document.find(f".//{NS}h1").find(f"{NS}p") is not None  # the unclosed <h1> swallows the paragraph

# Parse errors are recorded on the parser; strict mode raises the first one instead
parser = html5lib.HTMLParser()
parser.parse("<p>unclosed <b>bold</p>")
assert [code for _, code, _ in parser.errors] == ["expected-doctype-but-got-start-tag", "unexpected-end-tag"]
try:
    html5lib.HTMLParser(strict=True).parse("<p>unclosed <b>bold</p>")
except html5lib.html5parser.ParseError as e:
    print(f"strict mode: {e}")
else:
    raise AssertionError("strict parser accepted malformed HTML")

## Example 5: HTML fragment parsing and processing

In [ ]:
import html5lib

# Example 5: HTML fragment parsing and processing
print("Example 5: HTML fragment parsing and processing")

# Parse multiple HTML fragments
fragments = [
    "<p>Simple paragraph</p>",
    "<div class='container'><span>Nested content</span></div>",
    "<ul><li>List item 1</li><li>List item 2</li></ul>",
    "Plain text with <em>emphasis</em> and <strong>bold</strong>"
]

parsed_fragments = []
for i, fragment_html in enumerate(fragments):
    # Parse each fragment
    fragment = html5lib.parseFragment(fragment_html)
    parsed_fragments.append(fragment)
    
    print(f"Fragment {i+1}: {type(fragment)}")
    print(f"  Elements count: {len(list(fragment))}")

# Demonstrate fragment processing
print(f"\nProcessed {len(parsed_fragments)} fragments")

# Parse a complex fragment with nested elements
complex_fragment = """
<article>
    <header><h2>Article Title</h2></header>
    <section>
        <p>Article content with <a href="#">link</a></p>
        <blockquote>Quote content</blockquote>
    </section>
</article>
"""

complex_parsed = html5lib.parseFragment(complex_fragment)
print(f"Complex fragment parsed: {type(complex_parsed)}")

NS = "{http://www.w3.org/1999/xhtml}"
assert [len(list(f)) for f in parsed_fragments] == [1, 1, 1, 2]
# text before the first child element is kept on the fragment root
assert parsed_fragments[3].text == "Plain text with "
assert [c.tag for c in parsed_fragments[3]] == [f"{NS}em", f"{NS}strong"]
assert complex_parsed.find(f".//{NS}h2").text == "Article Title"
assert complex_parsed.find(f".//{NS}a").get("href") == "#"